# Free AI video clip on a Colab GPU (hero-shot test)

One click: **Runtime > Run all**. Needs a free GPU: *Runtime > Change runtime type > T4 GPU*. About 10-20 minutes the first time (model download), then 5-10 minutes per clip.

* Model: **Wan 2.1 text-to-video 1.3B** (open weights, runs in ~8 GB VRAM; the same family of models ComfyUI loads). Built with `diffusers` instead of a ComfyUI node graph so there is nothing to wire by hand.
* Output: a ~5 second 480p clip, shown below and saved as `clip.mp4` (download it from the Files panel).
* **Untested by the author** (no GPU where this was written). If a cell fails, send the error text back.
* AI video must be uploaded with YouTube's *altered or synthetic content* label. This notebook never uploads anything.

In [ ]:
#@title 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then Run all again"
print("GPU ok:", torch.cuda.get_device_name(0))

In [ ]:
#@title 2. Install (about 2 minutes)
!pip install -q -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg sentencepiece

In [ ]:
#@title 3. Load the model (first run downloads ~15 GB)
import torch
from diffusers import AutoencoderKLWan, WanPipeline
model_id = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(model_id, vae=vae, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()        # keeps a free 15 GB T4 from running out of memory
print("model loaded")

In [ ]:
#@title 4. Describe the shot and generate
prompt = "A friendly cartoon astronaut in a white suit walks across a purple alien planet under a starry sky, a ringed planet glowing in the distance, soft light, smooth camera follow, flat colourful animation style" #@param {type:"string"}
negative = "blurry, distorted, extra limbs, text, watermark, low quality, jittery" #@param {type:"string"}
seconds = 4 #@param {type:"slider", min:2, max:5, step:1}
fps = 15
frames = seconds * fps + 1           # Wan wants 4k+1 frames
frames = (frames - 1) // 4 * 4 + 1
import time; t0 = time.time()
out = pipe(prompt=prompt, negative_prompt=negative, height=480, width=832, num_frames=frames, guidance_scale=5.0, num_inference_steps=25).frames[0]
from diffusers.utils import export_to_video
export_to_video(out, "clip.mp4", fps=fps)
print(f"done in {time.time()-t0:.0f} s -> clip.mp4")

In [ ]:
#@title 5. Watch it, then download it
from IPython.display import Video, display
display(Video("clip.mp4", embed=True, width=480))
from google.colab import files
files.download("clip.mp4")